# Week 24, Day 1: Random walks and diffusion

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The square-root law

Simulate 10,000 walkers of 400 steps each at once: draw the steps as ±1 and take the cumulative sum along each row to get `X` (10,000 × 400). Compute the mean squared displacement at each step and fit a straight line to get `msd_slope`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r_w = np.random.default_rng(1964)
steps = r_w.choice([-1, 1], size=(10000, 400))
X = ...
msd = np.mean(X ** 2, axis=0)
n = np.arange(1, 401)
msd_slope = ...
print(f"variance at 100 steps {X[:, 99].var():.1f}, at 400 steps {X[:, 399].var():.1f};  MSD slope {msd_slope:.3f}")
print(f"mean distance from the start: {np.abs(X[:, 99]).mean():.2f} after 100 steps, {np.abs(X[:, 399]).mean():.2f} after 400")
plt.plot(n, msd); plt.xlabel("steps"); plt.ylabel("mean squared displacement"); plt.show()

In [ ]:
import numpy as np
def test_var():
    "the variance across walkers at step 400 is within 5% of 400"
    assert abs(X[:, 399].var() - 400) / 400 < 0.05

def test_slope():
    "the mean squared displacement grows with slope 1 per step (within 3%)"
    assert abs(msd_slope - 1) < 0.03

test_var()
test_slope()
print("All checks passed")

### Exercise: Spikes as first passages

Simulate 5,000 neurons at once from V = 0 with drift mu = 0.5 mV/ms and noise sigma = 2 mV/√ms (dt = 0.05 ms, up to 300 ms). Record for each the first time V reaches theta = 10 mV, and store these times in `fpt`.

In [ ]:
r_f = np.random.default_rng(1965)
mu, sigma, theta, dt = 0.5, 2.0, 10.0, 0.05
n_neur = 5000
V = np.zeros(n_neur); t_hit = np.full(n_neur, np.nan); alive = np.ones(n_neur, bool)
for k in range(1, int(300 / dt) + 1):
    V[alive] += ...
    crossed = alive & (V >= theta)
    t_hit[crossed] = k * dt
    alive &= ~crossed
    if not alive.any(): break
fpt = t_hit[~np.isnan(t_hit)]
print(f"{len(fpt)} spikes;  mean {fpt.mean():.2f} ms (theory 20), variance {fpt.var():.0f} (theory 320), median {np.median(fpt):.2f}, CV {fpt.std() / fpt.mean():.2f}")
plt.hist(fpt, 80, density=True); plt.xlabel("interspike interval (ms)"); plt.show()

In [ ]:
import numpy as np
def test_mean():
    "the mean first-passage time is within 3% of theta / mu = 20 ms"
    assert abs(fpt.mean() - 20) / 20 < 0.03

def test_var():
    "the variance is within 10% of theta sigma^2 / mu^3 = 320 ms^2"
    assert abs(fpt.var() - 320) / 320 < 0.10

def test_skew():
    "the distribution is right-skewed: the mean exceeds the median"
    assert fpt.mean() > np.median(fpt)

test_mean()
test_var()
test_skew()
print("All checks passed")

### Exercise: The heavy tail

Repeat with mu = 0 for 4,000 neurons and dt = 0.2 ms up to 1,600 ms. At each of t = 100, 200, 400, 800 and 1,600 ms record the fraction not yet absorbed in `surv`, and fit log surv against log t to get `surv_slope`.

In [ ]:
r_z = np.random.default_rng(1966)
dt0 = 0.2; n_z = 4000
V = np.zeros(n_z); alive = np.ones(n_z, bool)
checks = [100, 200, 400, 800, 1600]; surv = []
for k in range(1, int(1600 / dt0) + 1):
    V[alive] += sigma * np.sqrt(dt0) * r_z.normal(size=alive.sum())
    alive &= V < theta
    if any(abs(k * dt0 - c) < 1e-9 for c in checks):
        surv.append(...)
surv = np.array(surv)
surv_slope = ...
print("fraction still silent:", dict(zip(checks, [round(float(v), 3) for v in surv])), f";  slope {surv_slope:.3f}")

In [ ]:
import numpy as np
def test_slope():
    "the survival probability falls with a log-log slope within 0.08 of -0.5"
    assert abs(surv_slope + 0.5) < 0.08

def test_all():
    "survival is still above 10% at 1,600 ms"
    assert surv[-1] > 0.10

test_slope()
test_all()
print("All checks passed")